# SupportIQ — Fine-tuning XLM-RoBERTa multi-têtes (S3-J2, v2)

Encodeur **XLM-RoBERTa-base** partagé + **3 têtes** (catégorie / priorité / sentiment).
**v2 — passe propre sur le sentiment** : split de **validation** (jamais le test gelé),
**mean-pooling**, perte sentiment **pondérée**, **best-checkpoint** sur la validation.

- **Kaggle** : *Settings* → **GPU T4** + **Internet On**. Datasets ajoutés en *Input*.
- **Colab** : *Exécution > GPU (T4)*.

Baselines (macro-F1, S3-J1) : catégorie **0.91**, priorité **0.40**, sentiment **0.45** (TF-IDF).
v1 (CLS, 5 ep) : 0.91 / 0.35 / 0.60.

## 1. GPU + dépendances

In [ ]:
!nvidia-smi -L
!pip -q install "transformers>=4.44" "onnx>=1.16" "onnxruntime>=1.18"
print("OK")

## 2. Charger les données (Kaggle ou Colab)

In [ ]:
import glob, json

def find_data(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) or glob.glob(name)
    if hits:
        return hits[0]
    from google.colab import files
    print(f"Uploade {name} …"); files.upload()
    return name

train = [json.loads(l) for l in open(find_data("train.jsonl"), encoding="utf-8") if l.strip()]
test  = [json.loads(l) for l in open(find_data("test.jsonl"),  encoding="utf-8") if l.strip()]
print(f"train={len(train)}  test={len(test)}")

LABELS = {"category":["TECHNIQUE","FACTURATION","COMPTE","RECLAMATION","DEMANDE"],
          "priority":["LOW","MEDIUM","HIGH"], "sentiment":["NEG","NEU","POS"]}
TASKS = list(LABELS)
label2id = {t:{l:i for i,l in enumerate(v)} for t,v in LABELS.items()}

## 3. Tokenisation (XLM-R)

In [ ]:
import torch
from transformers import AutoTokenizer

MODEL="xlm-roberta-base"; MAX_LEN=256
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def encode(rows):
    enc = tokenizer([r["text"] for r in rows], truncation=True, padding="max_length",
                    max_length=MAX_LEN, return_tensors="pt")
    ys = {t: torch.tensor([label2id[t][r[t]] for r in rows]) for t in TASKS}
    return enc, ys

## 4. Modèle (mean-pooling) + split de validation + poids

- **Mean-pooling** : moyenne des tokens pondérée par le masque (au lieu du seul `<s>`).
- **Validation** : 15 % du train, stratifié sur le sentiment. On sélectionne le modèle dessus —
  **jamais sur le test**, qui reste gelé.

In [ ]:
import random, copy
from collections import Counter
import torch.nn as nn
from transformers import AutoModel

class MultiHead(nn.Module):
    def __init__(self, name, n_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.heads = nn.ModuleDict({t: nn.Linear(h, n) for t, n in n_labels.items()})
    def forward(self, input_ids, attention_mask):
        hs = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        pooled = (hs * m).sum(1) / m.sum(1).clamp(min=1e-9)   # moyenne masquée
        pooled = self.dropout(pooled)
        return {t: head(pooled) for t, head in self.heads.items()}

random.seed(42)
buckets = {}
for r in train: buckets.setdefault(r["sentiment"], []).append(r)
tr_split, val_split = [], []
for rows in buckets.values():
    rows = rows[:]; random.shuffle(rows)
    k = max(1, int(0.15*len(rows)))
    val_split += rows[:k]; tr_split += rows[k:]
random.shuffle(tr_split); random.shuffle(val_split)
print(f"train'={len(tr_split)}  val={len(val_split)}  test={len(test)}")

cnt = Counter(r["sentiment"] for r in tr_split); Ntot = sum(cnt.values())
device = "cuda" if torch.cuda.is_available() else "cpu"
sent_w = torch.tensor([Ntot/(3*cnt[l]) for l in LABELS["sentiment"]], dtype=torch.float, device=device)
print("poids sentiment:", {l: round(w,2) for l,w in zip(LABELS['sentiment'], sent_w.tolist())})

tr_enc, tr_y   = encode(tr_split)
val_enc, val_y = encode(val_split)

## 5. Entraînement (sentiment pondéré, best-checkpoint sur la validation)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, classification_report

model = MultiHead(MODEL, {t: len(v) for t, v in LABELS.items()}).to(device)
dl = DataLoader(TensorDataset(tr_enc["input_ids"], tr_enc["attention_mask"],
                              tr_y["category"], tr_y["priority"], tr_y["sentiment"]),
                batch_size=16, shuffle=True)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
ce = nn.CrossEntropyLoss(); ce_sent = nn.CrossEntropyLoss(weight=sent_w)
EPOCHS = 8

@torch.no_grad()
def sent_f1(enc, rows):
    model.eval(); p = []
    for i in range(0, len(enc["input_ids"]), 32):
        ids = enc["input_ids"][i:i+32].to(device); m = enc["attention_mask"][i:i+32].to(device)
        p += model(ids, m)["sentiment"].argmax(1).cpu().tolist()
    g = [label2id["sentiment"][r["sentiment"]] for r in rows]
    return f1_score(g, p, average="macro")

best, best_state = -1.0, None
for ep in range(EPOCHS):
    model.train(); tot = 0.0
    for ids, mask, yc, yp, ys in dl:
        ids, mask = ids.to(device), mask.to(device)
        yc, yp, ys = yc.to(device), yp.to(device), ys.to(device)
        lo = model(ids, mask)
        loss = ce(lo["category"], yc) + ce(lo["priority"], yp) + ce_sent(lo["sentiment"], ys)
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
    v = sent_f1(val_enc, val_split)
    print(f"epoch {ep+1}/{EPOCHS}  loss={tot/len(dl):.3f}  val_sentiment_F1={v:.3f}")
    if v > best: best, best_state = v, copy.deepcopy(model.state_dict())
model.load_state_dict(best_state)
print(f"meilleur checkpoint (validation): sentiment macro-F1={best:.3f}")

## 6. Évaluation FINALE sur le test set gelé (une seule fois)

In [ ]:
@torch.no_grad()
def predict(enc, bs=32):
    model.eval(); out = {t: [] for t in TASKS}
    for i in range(0, len(enc["input_ids"]), bs):
        ids = enc["input_ids"][i:i+bs].to(device); m = enc["attention_mask"][i:i+bs].to(device)
        lo = model(ids, m)
        for t in TASKS: out[t] += lo[t].argmax(1).cpu().tolist()
    return out

te_enc, _ = encode(test)
pred = predict(te_enc)
BASE = {"category":0.91,"priority":0.40,"sentiment":0.45}   # TF-IDF S3-J1
V1   = {"category":0.91,"priority":0.35,"sentiment":0.60}   # fine-tuné v1 (CLS)

print("=== v2 vs v1 vs baseline TF-IDF (macro-F1) ===")
for t in TASKS:
    g = [label2id[t][r[t]] for r in test]
    f1 = f1_score(g, pred[t], average="macro")
    print(f"{t:9} v2={f1:.2f}  v1={V1[t]:.2f}  TFIDF={BASE[t]:.2f}")
    print(classification_report(g, pred[t], target_names=LABELS[t], digits=2, zero_division=0))

## 7. Export ONNX (mean-pooling, exporteur stable)

In [ ]:
class ExportWrapper(nn.Module):
    def __init__(self, m):
        super().__init__(); self.enc = m.encoder; self.heads = m.heads
    def forward(self, input_ids, attention_mask):
        hs = self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        m = attention_mask.unsqueeze(-1).float()
        pooled = (hs * m).sum(1) / m.sum(1).clamp(min=1e-9)
        return self.heads["category"](pooled), self.heads["priority"](pooled), self.heads["sentiment"](pooled)

export_model = ExportWrapper(model).eval().cpu()
dummy = tokenizer("exemple de ticket", return_tensors="pt", padding="max_length", max_length=MAX_LEN)
torch.onnx.export(
    export_model, (dummy["input_ids"], dummy["attention_mask"]), "triage_xlmr.onnx",
    input_names=["input_ids","attention_mask"], output_names=["category","priority","sentiment"],
    dynamic_axes={"input_ids":{0:"batch",1:"seq"},"attention_mask":{0:"batch",1:"seq"},
                  "category":{0:"batch"},"priority":{0:"batch"},"sentiment":{0:"batch"}},
    opset_version=14, dynamo=False)
print("ONNX exporté : triage_xlmr.onnx")

## 8. Parité ONNX

In [ ]:
import onnxruntime as ort
sess = ort.InferenceSession("triage_xlmr.onnx", providers=["CPUExecutionProvider"])
outs = sess.run(None, {"input_ids": te_enc["input_ids"].numpy(),
                       "attention_mask": te_enc["attention_mask"].numpy()})
for i, t in enumerate(TASKS):
    g = [label2id[t][r[t]] for r in test]
    print(f"{t:9} ONNX macro-F1={f1_score(g, outs[i].argmax(1), average='macro'):.2f}")

## 9. Récupérer le modèle

In [ ]:
import os
tokenizer.save_pretrained("triage_tokenizer")
!zip -qr triage_model.zip triage_xlmr.onnx triage_tokenizer
if os.path.isdir("/kaggle/working"):
    print("Kaggle : triage_model.zip dans /kaggle/working -> onglet 'Output'.")
else:
    from google.colab import files; files.download("triage_model.zip")
print("Dézippe dans D:/Projects/StageProxym/ml/artifacts/ (gitignoré).")